# Lab 10: CNN Classification and Learned Representations

**STAT 486 · Fall 2026 · Estimated student work: 3–4 hours**

Fashion-MNIST is a dataset of grayscale images representing ten types of clothing and accessories. The dataset contains 60,000 training images and 10,000 test images, each of size 28×28 pixels. Each image is represented as a matrix of pixel intensity values ranging from 0 (black) to 255 (white). The ten classes are: T-shirt, Trouser, Pullover, Dress, Coat, Sandal, Shirt, Sneaker, Bag, and Ankle boot. This lab splits the official 60,000 training images into 50,000 for training and 10,000 for validation, using stratified random sampling to maintain class balance. The official 10,000 test images are reserved for final evaluation after all model choices are fixed.

Complete every **Task** in the code and Markdown cells immediately following it. API hints identify useful tools without implementing your analysis.


## Workflow and assessment

| Section | Evidence to produce | Points |
| --- | --- | ---: |
| 1. Data and baseline | Image inspection, RF results, split explanation | 5 |
| 2. CNN implementation | Model, `encode`, shape checks, explanation | 8 |
| 3. Training and validation | Loops, history, selected checkpoint, loss plot, comparison | 9 |
| 4. Confidence | Two ranked image grids and interpretation | 5 |
| 5. Learned representations | Aligned embeddings, two projections, error overlays, interpretation | 6 |
| 6. Final comparison | Test results and synthesis | 3 |
| Reproducibility | Environment files and an executable completed notebook | 4 |
| **Total** | | **40** |

Use the validation set for checkpoint selection and Sections 3–5. Keep the official test set for Section 6, after all model choices and diagnostic interpretations are complete. Use the same data split for both models.

One eight-epoch CNN run is required; no hyperparameter search is required. Correct procedure and supported interpretation matter more than reaching a particular accuracy. PCA is a linear projection; t-SNE emphasizes local neighborhoods. Neither plot supplies a causal explanation for a prediction.


## 0. Setup

Select this repository's `.venv` kernel before running the setup cells. The README explains `uv sync`. The required work supports a local CPU; compatible accelerators are used if available. Fixed seeds help reproduce a run in the same environment, but results can differ across hardware and library implementations.

The CPU thread limit below keeps a small model from oversubscribing laptop CPUs. `num_workers=0` avoids notebook multiprocessing problems on Windows. Starter task cells contain comments so the uncompleted starter can execute its supplied setup; implement each task before running your completed notebook from a fresh kernel.


In [ ]:
import os
import random
from copy import deepcopy
from pathlib import Path
from time import perf_counter

import matplotlib.pyplot as plt
import numpy as np
import torch
from matplotlib.lines import Line2D
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.manifold import TSNE
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision.datasets import FashionMNIST
from torchvision.transforms import ToTensor

SEED = 486
BATCH_SIZE = 128
EPOCHS = 8
LEARNING_RATE = 0.001

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(max(1, min(4, os.cpu_count() or 1)))

if torch.cuda.is_available():
    device = torch.device("cuda")
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"PyTorch {torch.__version__}; device={device}; CPU threads={torch.get_num_threads()}")


### Data source and split

The next cell downloads approximately 31 MB of compressed data on its first run, then reuses the local `data/` cache. It uses a pinned revision of the [official Fashion-MNIST repository](https://github.com/zalandoresearch/fashion-mnist/tree/b2617bb6d3ffa2e429640350f613e3291e10b141/data/fashion). Torchvision verifies each archive against its expected checksum. The supplied mirror setting keeps the files tied to that revision.

The official 60,000-image training set is split into 50,000 training images and 10,000 validation images, stratified by class. The official 10,000-image test set is reserved for the final section. Downloading test files during setup does not permit using their results to change the model.

`ToTensor()` converts each image to a float tensor of shape `(1, 28, 28)` with pixels in `[0, 1]`. No estimated normalization parameters or augmentation are needed. `train_indices` and `val_indices` refer to rows in the original official training dataset; `val_ids` preserves that mapping.


In [ ]:
DATA_DIR = Path("data")
DATA_REVISION = "b2617bb6d3ffa2e429640350f613e3291e10b141"
FashionMNIST.mirrors = [
    f"https://raw.githubusercontent.com/zalandoresearch/fashion-mnist/{DATA_REVISION}/data/fashion/"
]

full_train = FashionMNIST(DATA_DIR, train=True, download=True, transform=ToTensor())
test_data = FashionMNIST(DATA_DIR, train=False, download=False, transform=ToTensor())
class_names = full_train.classes

train_indices, val_indices = train_test_split(
    np.arange(len(full_train)),
    test_size=10_000,
    stratify=full_train.targets.numpy(),
    random_state=SEED,
)
val_ids = val_indices.copy()
y_train = full_train.targets[train_indices].numpy()
y_val = full_train.targets[val_indices].numpy()

train_data = Subset(full_train, train_indices)
val_data = Subset(full_train, val_indices)
train_loader = DataLoader(
    train_data,
    batch_size=BATCH_SIZE,
    shuffle=True,
    generator=torch.Generator().manual_seed(SEED),
    num_workers=0,
)
val_loader = DataLoader(val_data, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

assert len(train_data) == 50_000 and len(val_data) == 10_000
assert len(test_data) == 10_000
assert np.intersect1d(train_indices, val_indices).size == 0
print(f"Training: {len(train_data):,}; validation: {len(val_data):,}; reserved test: {len(test_data):,}")
print(dict(enumerate(class_names)))


## 1. Data inspection and a random forest baseline

### Task 1.1: Inspect the training data

Display one **training** image from each class in a labeled grid. Report one image tensor's shape and dtype and show the training and validation class counts. Use readable class names throughout the lab.

**Hints:** a `Subset` contains `(image, label)` pairs; `np.bincount` counts integer labels; `imshow(..., cmap="gray")` displays a grayscale image.


In [ ]:
# Your code here.
# Labeled training-image grid, tensor shape/dtype, and class counts.


### Task 1.2: Explain the split

In two or three sentences, explain the distinct roles of training, validation, and test data in this lab. Why must both models use the same training and validation observations?


*Write your response here.*


### Task 1.3: Fit the baseline

Construct `X_train_rf` and `X_val_rf` from image rows selected by `train_indices` and `val_indices`. Each row must contain 784 flattened pixels, converted to `float32` and divided by 255 to match the CNN's pixel scale. Use the supplied `y_train` and `y_val`.

Fit a `RandomForestClassifier` with **100 trees**, `random_state=SEED`, and `n_jobs=-1`; leave other parameters at their defaults. Store the fitted model as `rf`, elapsed fitting time as `rf_fit_seconds`, validation predictions as `rf_val_pred`, and accuracy as `rf_val_accuracy`. Report fitting time and validation accuracy clearly.

**Hints:** access images through `full_train.data`; flatten with `reshape`; use `perf_counter()` immediately before and after fitting, and `accuracy_score` for accuracy. Do not fit on validation data. Free the flattened training matrix after fitting if memory is limited.


In [ ]:
# Your code here.

# X_train_rf: (50_000, 784), float32 pixels in [0, 1]
# X_val_rf: (10_000, 784), in exactly the y_val / val_ids row order
# Required outputs: rf, rf_fit_seconds, rf_val_pred, rf_val_accuracy


### Task 1.4: Interpret the baseline

Report the baseline's validation accuracy and fitting time in a sentence. Explain one difference between how a random forest on flattened pixels and a CNN can use image structure.


*Write your response here.*


## 2. Implement the CNN

### Task 2.1: Build the prescribed model

Implement a class named `FashionCNN(nn.Module)` with the architecture below. Initialize every trainable layer from scratch.

| Operation | Output shape for a batch of size B |
| --- | --- |
| Input | `(B, 1, 28, 28)` |
| `Conv2d`: 1 input channel, 16 output channels, kernel 3, padding 1; ReLU; max pool 2 | `(B, 16, 14, 14)` |
| `Conv2d`: 16 input channels, 32 output channels, kernel 3, padding 1; ReLU; max pool 2 | `(B, 32, 7, 7)` |
| Flatten, preserving the batch dimension | `(B, 1568)` |
| `Linear`: 1568 inputs, 64 outputs; ReLU | `(B, 64)` |
| `Linear`: 64 inputs, 10 outputs | `(B, 10)` |

Implement two methods:

- `encode(x)` returns the **64 activations after the hidden linear layer's ReLU**.
- `forward(x)` applies the final classification layer to those activations and returns **raw logits**.

This defines "penultimate linear layer embedding" here: the hidden representation fed into the final classifier, rather than its weights or its ten class scores. Reuse `encode` inside `forward` so both methods use the same feature calculations.

**Hints:** use `super().__init__()`, `nn.Sequential`, `nn.Conv2d`, `nn.ReLU`, `nn.MaxPool2d`, `nn.Flatten`, and `nn.Linear`. Do not apply softmax inside the model; cross-entropy expects raw logits.


In [ ]:
# Your code here.

# class FashionCNN(nn.Module):
#     def __init__(self):
#         ...
#
#     def encode(self, x):
#         ...
#
#     def forward(self, x):
#         ...


### Task 2.2: Check the model before training

Instantiate the model on `device`, pass a small batch from `train_loader` through both methods, and check that logits have shape `(B, 10)` and embeddings have shape `(B, 64)`. Verify that labels have integer dtype `torch.int64` and values from 0 through 9. Show the model and check results.

Why is the output dimension 10? Why do we retain the batch dimension when flattening? Answer briefly below.


In [ ]:
# Your code here.

# Instantiate, inspect, and assert expected shapes and label dtype.
# A batch is obtained with next(iter(train_loader)).
# Move the input batch to device before passing it to the model.


*Write your response here.*


## 3. Train, validate, and select a checkpoint

### Task 3.1: Implement one epoch

Implement `run_epoch(model, loader, loss_fn, optimizer=None)` returning `(mean_loss, accuracy)`. An optimizer means training; `optimizer=None` means evaluation.

Your function must:

1. Select `model.train()` or `model.eval()` appropriately.
2. Move input images and integer labels to `device`.
3. Compute cross-entropy from **raw logits** and labels.
4. During training only: clear old gradients, backpropagate, and update parameters.
5. Disable gradient tracking during evaluation and never update parameters there.
6. Accumulate loss as `batch_mean_loss * actual_batch_size`, then divide by the total number of examples. Compute accuracy from total correct predictions divided by total examples.

The last batch may be shorter than 128, so averaging batch means equally gives the wrong weighting.

**Hints:** `torch.set_grad_enabled(training)` can surround the loop; `optimizer.zero_grad()`, `loss.backward()`, and `optimizer.step()` implement an update. `logits.argmax(dim=1)` supplies predictions. See the [PyTorch optimization tutorial](https://docs.pytorch.org/tutorials/beginner/basics/optimization_tutorial.html) and [cross-entropy reference](https://docs.pytorch.org/docs/2.9/generated/torch.nn.CrossEntropyLoss.html).


In [ ]:
# Your code here.

# def run_epoch(model, loader, loss_fn, optimizer=None):
#     # Decide mode and initialize example-weighted totals.
#     # Enable gradients only when training.
#     # Loop over batches, compute outputs/loss, and optionally update parameters.
#     # Accumulate totals and return mean loss and accuracy.
#     ...


### Task 3.2: Train for eight epochs

Create a **fresh** `FashionCNN` named `model`, a `CrossEntropyLoss`, and Adam with `lr=LEARNING_RATE`. Reset the PyTorch seed to `SEED` before initializing this model and reset the training loader's generator seed so earlier batch inspection does not change the training order.

Train for `EPOCHS` (eight). At every epoch, run training and then validation. Store a `history` list with one dictionary per epoch containing `epoch`, `train_loss`, `val_loss`, `train_accuracy`, and `val_accuracy`. Print a readable epoch summary.

Keep `best_epoch`, `best_val_loss`, and a **deep copy** of the model's `state_dict()` whenever validation loss reaches a new minimum. A shallow reference to `state_dict()` can keep changing as training continues. After eight epochs, restore the best state and evaluate it to obtain `cnn_val_loss` and `cnn_val_accuracy`.

Record the elapsed eight-epoch training procedure as `cnn_fit_seconds`, including per-epoch validation passes. Report the chosen epoch, its validation loss and accuracy, and elapsed time. Keep this restored model unchanged for every remaining section.

**Hints:** `train_loader.generator.manual_seed(SEED)`, `deepcopy`, and `model.load_state_dict(...)` are useful. Select by lowest validation **loss**, even if another epoch has higher validation accuracy. Keep the checkpoint in memory; no model file is required.


In [ ]:
# Your code here.

# Fresh initialization, optimizer/loss, epoch loop, history, copied best checkpoint,
# restoration, and selected-model validation metrics.
# Required outputs: model, history, best_epoch, best_val_loss,
#                   cnn_fit_seconds, cnn_val_loss, cnn_val_accuracy


### Task 3.3: Plot and interpret validation behavior

Plot **training loss and validation loss on the same axes**, with labeled axes, a legend, and the selected epoch marked. Use stored history, rather than rerunning training.

Display a validation comparison of RF and the selected CNN, including accuracy and fitting/training time. Label that CNN timing includes validation passes, so this is a practical runtime comparison rather than a controlled speed benchmark.


In [ ]:
# Your code here.
# Loss curves with selected epoch marked, and validation comparison.


Answer using your actual curves and metrics:

1. Which epoch was selected, and how does its validation loss compare with the final epoch? Why reload a checkpoint instead of automatically using final weights?
2. Do the curves show evidence of overfitting, underfitting, or neither clearly? Cite a specific feature; improvement need not be monotonic.
3. How did the CNN compare with RF on validation accuracy and runtime?


*Write your response here.*


## 4. Inspect confident predictions

### Task 4.1: Collect ordered validation predictions

Using the **restored best model**, run inference over `val_loader` in evaluation mode with gradients disabled. Concatenate logits on the CPU into a tensor `val_logits` of shape `(10_000, 10)` and collected labels into a NumPy array `val_labels`.

Check that `val_labels` exactly equals the supplied `y_val`. This verifies that predictions refer to the same ordered images. Do not shuffle the validation loader.

Apply softmax **across the class dimension**. For each row, confidence is the largest softmax probability, and the predicted class is its index:

\[
p_{ik}=\frac{\exp(z_{ik})}{\sum_{j=1}^{10}\exp(z_{ij})},
\qquad
\hat y_i=\arg\max_k p_{ik},
\qquad
c_i=\max_k p_{ik}.
\]

Store `val_probs` (NumPy, shape `(10_000, 10)`), `val_pred`, `val_confidence`, and the Boolean mask `val_correct`. Check probability bounds, row sums close to one, and agreement with `cnn_val_accuracy`.

Rank **correct and incorrect predictions separately** in descending confidence. Store up to nine validation row positions in `top_correct` and `top_incorrect`. Break confidence ties by row position for reproducible displays.

**Hints:** use `torch.inference_mode()` or `torch.no_grad()`, `torch.cat`, `torch.softmax`, `np.flatnonzero`, and a stable `np.argsort`. All row positions here are within the validation set, not original Fashion-MNIST row IDs.


In [ ]:
# Your code here.

# Ordered logits/labels, softmax, predictions/confidence, checks,
# and independently ranked top_correct / top_incorrect row positions.


### Task 4.2: Display both groups

Produce two titled image grids: **most confident incorrect predictions** and **most confident correct predictions**. For each image, show:

- validation row position;
- true and predicted class names; and
- softmax confidence with at least four decimal places.

Retrieve the original image for validation position `row` with `full_train.data[val_ids[row]]`; do not use `row` directly to index the original training dataset.

Show nine images per group when available. If fewer than nine qualify, display available images and hide unused axes. If none qualify, print a clear message instead of failing. Preserve `top_incorrect` for the next section.


In [ ]:
# Your code here.
# Two image grids with correctly aligned labels and confidences.


### Task 4.3: Interpret confidence

Identify at least two confident mistakes by validation row position and describe what you can see in those images. Compare them with confident correct examples. Does a large softmax probability guarantee a correct or calibrated probability estimate? Support your answer with your results; no calibration experiment is required.

If fewer than two mistakes exist, discuss those available and state that limitation.


*Write your response here.*


## 5. Visualize the learned representation

### Task 5.1: Extract the full validation embedding

Use `model.encode` on **all** validation images with the same restored checkpoint, evaluation mode, disabled gradients, and unshuffled `val_loader`. Concatenate CPU results into a NumPy array `val_embeddings` of shape `(10_000, 64)`.

Collect labels again and verify exact agreement with `val_labels`. Check the embedding shape and that all entries are finite. Keep every row aligned with `val_pred`, `val_confidence`, `val_correct`, and `val_ids`.

These are activations for each image, not the layer's learned weight matrix. Do not retrain the CNN or use ten-dimensional logits as the embedding.


In [ ]:
# Your code here.
# Full ordered val_embeddings plus shape, finite, and alignment checks.


### Task 5.2: Compute independent two-dimensional projections

Compute both projections from the **same full, unscaled `val_embeddings` array**:

- PCA: two components with `svd_solver="full"`. Store the fitted object as `pca`, coordinates as `pca_2d`, and report the explained-variance ratio for each component and their sum.
- t-SNE: `n_components=2`, `perplexity=30`, `init="random"`, `learning_rate="auto"`, `max_iter=1000`, `method="barnes_hut"`, and `random_state=SEED`. Store coordinates as `tsne_2d` and report elapsed t-SNE time.

Each output must have shape `(10_000, 2)` and contain finite values. PCA centers internally; do not add feature standardization or whitening in this lab.

**Independently means:** fit PCA directly to the 64-dimensional embedding, and fit t-SNE directly to that same embedding. Do not feed two-dimensional PCA coordinates into t-SNE. This is exploratory validation-set visualization, not a new classifier or a test-performance estimate.

Full-validation t-SNE may take several minutes on a CPU. Run it once, retain its coordinates, and reuse them while adjusting styling. See the [scikit-learn t-SNE reference](https://scikit-learn.org/1.7/modules/generated/sklearn.manifold.TSNE.html).


In [ ]:
# Your code here.

# Independently fit pca_2d and tsne_2d from val_embeddings,
# report PCA variance and t-SNE elapsed time, and check both output arrays.


### Task 5.3: Make side-by-side plots and link errors

Create one figure with **PCA on the left and t-SNE on the right**.

- Plot all 10,000 validation examples, colored by **true class**, using the same ten discrete colors in both panels.
- Add a readable class legend, titles, and axis labels (`PC 1`/`PC 2` and `t-SNE 1`/`t-SNE 2`).
- Draw misclassified examples again with larger hollow black circles so true-label colors remain visible. Include an error-marker legend.
- Annotate the same **three highest-confidence incorrect examples** (`top_incorrect[:3]`) with validation row positions in both panels. If fewer exist, annotate those available.
- Use small points, transparency, and a sufficiently large figure so the full validation set remains readable.

Do not force identical numerical limits across PCA and t-SNE: their coordinate scales differ.

**Hints:** `plt.subplots(1, 2, ...)`, a discrete palette such as `tab10`, `facecolors="none"`, `edgecolors="black"`, `ax.annotate`, and `Line2D` legend handles are useful.


In [ ]:
# Your code here.

# Side-by-side full-validation plots, true-label colors, all-error overlays,
# and matching annotations for the top three confident errors.


### Task 5.4: Interpret what the plots show

Write one or two short paragraphs addressing:

1. Which classes appear separated or overlapping in each projection? Cite at least two class names and compare PCA with t-SNE.
2. Where do highlighted errors appear relative to their true and predicted classes? Connect the three annotated points to their images in Section 4. State when a projection does not offer a convincing visual explanation.
3. How much variance do the first two PCs retain? What information might a two-dimensional projection discard?
4. Why should t-SNE axes, apparent cluster sizes, and distances between distant groups be interpreted cautiously? Why does apparent overlap or separation not prove the reason for a CNN error?

Use observations from your run, including counterexamples if errors appear inside apparently separated groups. A plausible pattern is a hypothesis, not proof of causation.


*Write your response here.*


## 6. Final test comparison

### Task 6.1: Evaluate the fixed models once

Complete the preceding interpretation before evaluating the test set. Keep the RF fitted in Section 1 and CNN checkpoint selected in Section 3; do not retrain on combined training and validation data.

Flatten and scale official test images using the same RF preprocessing. Obtain RF test accuracy. Evaluate the CNN on `test_loader` in evaluation mode with gradients disabled and obtain test accuracy. Run this final evaluation once after model choices are fixed; do not use scores to tune or select either model.

Display a labeled comparison table with rows for RF and CNN and columns for validation accuracy, test accuracy, and fitting/training seconds. State that CNN time includes per-epoch validation passes. Report accuracies consistently as proportions or percentages.

**Hints:** `test_data.data` and `test_data.targets` expose official test arrays; `run_epoch(..., optimizer=None)` can evaluate the CNN. A Markdown table printed with `print` or displayed with IPython is sufficient; pandas is not required.


In [ ]:
# Your code here.

# Final RF/CNN test accuracy and comparison table.
# Required outputs: rf_test_accuracy, cnn_test_accuracy


### Task 6.2: Synthesize the evidence

In one short paragraph, compare validation and test performance and practical runtime. What did the confident-example inspection and embedding plots add beyond accuracy? Identify one limitation and one sensible follow-up experiment with a new evaluation protocol. Describe the follow-up; do not run it on the test set.


*Write your response here.*


## Completion check

Before submitting:

- Complete every code task and replace every written-response placeholder.
- Restart the kernel and run the completed notebook from top to bottom; save it **with required outputs and plots**.
- Confirm the loss plot, two confidence grids, full-validation side-by-side projections, and comparison table are visible and readable.
- Keep the supplied `pyproject.toml` and `.python-version`; commit generated `uv.lock` with the completed notebook.
- Keep downloaded data, `.venv`, caches, and model files out of Git. Data is downloaded automatically.
- Follow the README to submit your private repository's root URL to **Canvas Lab 10**.

No saved model, separate answer sheet, or additional tuning experiment is required.
